# FireBench 데이터셋 분석
Google Research의 FireBench는 TPU 기반 고성능 산불 시뮬레이션 데이터셋입니다.

1. **데이터셋 생성 방식**: SWIRL-FIRE 프레임워크, 200k TPU 시간, 1.36 PiB 규모
2. **목적**: 기후-화재 상호작용, ML 모델 개발

In [ ]:
# 1. 실행 환경 및 FireBench 로더 준비
import sys
import os
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Tuple, List, Optional
import re

# 프로젝트 루트에서 실행할 때 firespread 폴더를 모듈 경로에 추가
sys.path.insert(0, "firespread")

try:
    import gcsfs
    fs = gcsfs.GCSFileSystem(token="anon")
    print("✅ gcsfs 준비 완료 (GCS 접근 가능)")
except ImportError:
    print("⚠️ gcsfs 미설치. GCS 접근 불가능")
    fs = None

# FireBench 로더 함수 임포트
from firebench_loader import (
    extract_slope_from_path,
    _ramp_dirs,
    load_firebench_sample_from_path,
    _wind_stats,
    _spread_stats,
    _open_dataset,
)

print("✅ FireBench 로더 준비 완료")

## GCS에서 FireBench 시뮬레이션 경로 수집

공개 경로는 `u{풍속}/ramp{경사각}/fire.zarr` 구조입니다. 예를 들어 `u6/ramp15/fire.zarr`는 풍속 6 m/s, 경사각 15° 조건을 나타냅니다.

In [ ]:
# 2. GCS 경로 수집 (시간이 소요되므로 제한적으로 수집)
print("🔍 FireBench GCS 구조 탐색 중...\n")

if fs is not None:
    try:
        prefix = "firebench/v2024.04"
        ramp_directories = _ramp_dirs(prefix)
        print(f"📦 발견된 시뮬레이션 디렉터리: {len(ramp_directories)}")
        print("\n첫 10개 경로:")
        for i, ramp_dir in enumerate(ramp_directories[:10]):
            slope_deg = extract_slope_from_path(ramp_dir)
            print(f"  {i+1}. {ramp_dir} → slope={slope_deg}°")
    except Exception as e:
        print(f"❌ GCS 접근 오류: {e}")
else:
    print("⚠️ GCS 접근 불가능 - 로컬 분석만 진행합니다")

## 경로에서 제공되는 메타데이터

실제 FireBench 경로 예: `firebench/v2024.04/u6/ramp15/fire.zarr`

- **풍속**: `u6`처럼 `u` 뒤 경로 요소에서 제공
- **경사각**: `ramp15`처럼 `ramp` 뒤 숫자로 제공
- **경사 방위각(aspect)**: 이 경로 구조만으로는 알 수 없음.

In [ ]:
# 3. 실제 경로에서 풍속 조건과 경사각 추출 테스트
sample_paths = [
    "firebench/v2024.04/u6/ramp0/fire.zarr",
    "firebench/v2024.04/u6/ramp15/fire.zarr",
    "firebench/v2024.04/u10/ramp30/fire.zarr",
]

print("경로 메타데이터 추출 테스트:\n")
print(f"{'경로':<50} | {'경사각(°)':<10}")
print("-" * 65)

for path in sample_paths:
    slope_deg = extract_slope_from_path(path)
    print(f"{path:<50} | {slope_deg:<10.1f}")

## FireBench 실제 Zarr 구조 및 좌표

공개 표본 `u6/ramp15/fire.zarr`의 메타데이터를 확인한 결과:

- 차원 크기: `t=551`, `x=1528`, `y=252`, `z=3584`
- 공식 브라우저에서 공간 좌표 `x`, `y`, `z`는 미터 단위로 표시
- 표본의 좌표 증가량: `x=1 m`, `y=1 m`, `z=0.5 m`
- `t` 좌표 속성은 나노초이며, 연속 출력 간격은 `1 s` (내부 solver `dt=0.0025 s`와 구분)
- 주요 변수에는 `u`, `v`, `w`, `T_s`, `theta`, `rho_f` 등이 포함

전체 Zarr 배열은 매우 크므로 구조 확인은 메타데이터 조회로 하고, 필요한 변수·시점·공간 구간만 선택해야 합니다.

In [ ]:
# 4. 공개 FireBench 표본에서 확인한 Zarr 메타데이터
sample_metadata = {
    "표본": "gs://firebench/v2024.04/u6/ramp15/fire.zarr",
    "차원 크기": {"t": 551, "x": 1528, "y": 252, "z": 3584},
    "좌표 간격 (m)": {"x": 1.0, "y": 1.0, "z": 0.5},
    "출력 시간 간격": "1초 (t 좌표 nanoseconds 단위)",
    "solver dt": "0.0025초 (출력 간격과 다름)",
    "변수 예시": ["u", "v", "w", "T_s", "theta", "rho_f"],
    "방위각 메타데이터": "경로/전역 속성에서 확인되지 않음",
}

for key, value in sample_metadata.items():
    print(f"{key}: {value}")

print("\n전체 배열은 읽지 않고 공개 Zarr의 메타데이터만 조회한 결과입니다.")

## FireBench 실험 설계

FireBench는 제어된 조건에서 SWIRL-FIRE로 고해상도 화재 시뮬레이션을 생성합니다. 공개 경로에서 확인되는 주요 조건은 풍속(`u...`)과 경사각(`ramp...`)입니다.

- 공개 데이터셋은 여러 풍속·경사 조합으로 구성됩니다(소개 글 기준 117개 조합).
- 아래 셀의 표와 시각화는 설명용 예시이며 실제 FireBench 조합 개수를 나타내지 않습니다.

In [ ]:
# 5. 공개 경로에서 실제 풍속-경사 조합 확인
if fs is None:
    print("gcsfs가 없어 공개 조합을 조회할 수 없습니다.")
else:
    simulation_paths = _ramp_dirs("firebench/v2024.04")
    records = []
    for path in simulation_paths:
        match = re.search(r"/u([\d.]+)/ramp([\d.]+)/", path)
        if match:
            records.append({
                "wind_speed": float(match.group(1)),
                "slope_deg": float(match.group(2)),
            })

    combinations = pd.DataFrame(records)
    availability = pd.crosstab(combinations["slope_deg"], combinations["wind_speed"])
    print(f"실제 발견된 시뮬레이션 경로 수: {len(combinations)}")
    print("경로 메타데이터로부터 확인한 풍속-경사 조합 (aspect 정보는 없음):")
    display(availability)

    ax = sns.heatmap(availability, cmap="Blues", cbar_kws={"label": "시뮬레이션 수"})
    ax.set_title("FireBench 공개 경로의 풍속 × 경사각 조합")
    ax.set_xlabel("풍속 경로 값 (u, m/s)")
    ax.set_ylabel("경사각 (°)")
    plt.tight_layout()
    plt.show()

## 화재 확산 및 유동 특징량 계산

FireBench에서 추출되는 핵심 특징:
- **풍향, 풍속**: 대기 유동
- **확산 방향, 확산 속도**: 화재 진행
- **온도장**: 연직 분포

In [ ]:
# 6. 실제 loader의 특징 추출 흐름
print("FireBench 특징량 추출 파이프라인:\n")

feature_extraction = {
    "1단계": {
        "입력": ["u, v 속도 성분"],
        "방법": "공간 평균 및 atan2(v, u)",
        "출력": ["wind_dir_local (°)", "wind_speed (m/s)"],
    },
    "2단계": {
        "입력": ["연속 시점의 T_s"],
        "방법": "고온 중심 이동 + x/y 좌표 차이 + t 좌표 차이",
        "출력": ["spread_dir_local (°)", "spread_speed (m/s)"],
    },
    "3단계": {
        "입력": ["Zarr 경로"],
        "방법": "u{wind}/ramp{slope} 경로 파싱",
        "출력": ["wind metadata", "slope_deg (°)"],
    },
    "4단계": {
        "입력": ["로컬 풍향, 풍속, 경사각"],
        "방법": "NumPy 배열 결합",
        "출력": ["X = [wind_dir_local, wind_speed, slope_deg]", "y = [spread_dir_local, spread_speed]"],
    },
}

for step, details in feature_extraction.items():
    print(f"◆ {step}:")
    for key, value in details.items():
        print(f"   {key}: {', '.join(value) if isinstance(value, list) else value}")
    print()

print("📊 CSV 열 예시:")
sample_data = pd.DataFrame({
    "wind_dir_local": [45.0, 90.0, 135.0, 180.0],
    "wind_speed": [2.5, 4.3, 6.1, 8.2],
    "slope_deg": [0.0, 15.0, 30.0, 45.0],
    "spread_dir_local": [50.0, 92.0, 138.0, 175.0],
    "spread_speed": [0.3, 0.8, 1.5, 2.1],
})
print(sample_data.to_string(index=False))
print("\nX 특징: 3개 열 / y 목표: 확산 방향·속도 2개 열")